# 01 · Data Profiling — Diagnóstico da Base RAW

**Objetivo deste notebook:** entender, antes de qualquer limpeza, a real condição da
base de clientes consolidada a partir de diferentes sistemas de origem
(`data/raw/customers_raw.csv`). O profiling é o ponto de partida de qualquer
projeto de dados: ele orienta *quais* transformações de limpeza serão
necessárias no notebook seguinte (`02_data_cleaning.ipynb`).

Vamos analisar: dimensões, tipos de dados, valores ausentes, duplicidades,
cardinalidade e uma primeira leitura estatística (`describe`) — sempre
interpretando o que cada resultado significa para o negócio.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.data_loader import load_raw_data

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

df = load_raw_data()
df.head(10)


,Customer ID,Customer Name,Email,Age,Gender,City,State,Signup Date,Income,Purchase Count,Total Spent,Satisfaction Score
0,CUST001385,Emilly da Rocha,emilly.da.rocha@mail.com,52.0,Fem,Manaus,amazonas,2025/05/30,3181,7,"R$ 1.356,00",3.1
1,CUST000466,Bento Pastor,BENTO.PASTOR@CORREIO.NET,61.0,Masc,NaN,sp,17/03/2023,4.888,6,1724,5.0
2,CUST001552,Ana Vitória Cirino,ana.vitoria.cirino@webmail.com,29.0,Fem,Fortaleza,CE,2023/08/08,6726,14,"R$ 2.934,00",5.0
3,CUST003355,gael aragão,gael.aragao@correio.net,27.0,male,Uberlândia,mg,12/08/2023,"R$ 2.574,00",6,1448,3.8
4,CUST001680,Enrico Carvalho,ENRICO.CARVALHO@WEBMAIL.COM,23.0,male,Duque de Caxias,RJ,29-06-2024,NaN,8,"R$ 2.324,00",4.4
5,CUST000379,Anthony Martins,anthony.martins@correio.net,34.0,MASCULINO,Brasília,DF,2024-07-14,2643,7,1097,4.5
6,CUST001653,Ryan da Conceição,ryan.da.conceicao@provedor.com.br,26.0,Masc,Goiânia,Goiás,2024/10/27,1956,8,1883,4.8
7,CUST000436,Erick Camargo,ERICK.CAMARGO@PROVEDOR.COM.BR,30.0,Não binário,Contagem,MG,12-04-2024,NaN,7,1.713,2.9
8,CUST000412,Ana Luiza Porto,ana.luiza.porto@email.com,44.0,Fem,Feira de Santana,bahia,2024-01-09,2746,6,"R$ 714,00",3.5
9,CUST002550,Dr. Davi Rezende,dr..davi.rezende@mail.com,50.0,Masculino,Porto Alegre,rio grande do sul,30-03-2025,2895,14,2835,3.9


## Dimensões da base

Quantos registros e colunas temos? Este número ainda **não** representa a
quantidade real de clientes — como veremos adiante, a base contém duplicidades.


In [2]:
print(f"Linhas: {df.shape[0]:,}".replace(",", "."))
print(f"Colunas: {df.shape[1]}")


Linhas: 5.250
Colunas: 12


## Tipos de dados

Como o carregamento é feito preservando tudo como texto (`dtype=str`) — decisão
proposital em `data_loader.load_raw_data`, para não perder informação antes da
limpeza — **todas** as colunas aparecem como `object`, mesmo as que deveriam
ser numéricas (`Age`, `Income`, `Purchase Count`, `Total Spent`,
`Satisfaction Score`) ou de data (`Signup Date`). Isso já é, por si só, um
problema de qualidade a ser corrigido: valores numéricos armazenados como
texto não podem ser somados, comparados ou usados em modelos estatísticos sem
conversão explícita.


In [3]:
df.dtypes


Customer ID           str
Customer Name         str
Email                 str
Age                   str
Gender                str
City                  str
State                 str
Signup Date           str
Income                str
Purchase Count        str
Total Spent           str
Satisfaction Score    str
dtype: object

## Valores nulos

Contagem e percentual de valores ausentes por coluna.


In [4]:
nulos = df.isnull().sum()
percentual_nulos = (nulos / len(df) * 100).round(2)

resumo_nulos = pd.DataFrame({"missing_count": nulos, "missing_percentage": percentual_nulos})
resumo_nulos = resumo_nulos[resumo_nulos["missing_count"] > 0].sort_values("missing_count", ascending=False)
resumo_nulos


,missing_count,missing_percentage
Satisfaction Score,407,7.75
Income,308,5.87
City,255,4.86
Age,207,3.94


**Interpretação:** os valores ausentes se concentram em quatro colunas —
`Age`, `City`, `Income` e `Satisfaction Score` — com proporções entre ~4% e
~8%. Isso é consistente com uma base real vinda de múltiplos sistemas, onde
nem todo formulário de cadastro exige todos os campos. Como a proporção é
baixa a moderada, técnicas de imputação (mediana, mediana por grupo, ou
categoria "unknown") são apropriadas — **não** faremos `fillna(0)`, o que
distorceria a distribuição real dessas variáveis (ver notebook 02).


## Duplicidades

Duas visões: duplicidade **completa** (todas as colunas idênticas) e
duplicidade pela **chave de negócio** (`Customer ID`).


In [5]:
duplicidade_completa = df.duplicated().sum()
duplicidade_por_id = df.duplicated(subset=["Customer ID"], keep=False).sum()
ids_unicos = df["Customer ID"].nunique()

print(f"Linhas totais: {len(df):,}".replace(",", "."))
print(f"Duplicidades completas: {duplicidade_completa}")
print(f"Linhas envolvidas em duplicidade de Customer ID: {duplicidade_por_id}")
print(f"Customer IDs únicos: {ids_unicos:,}".replace(",", "."))
print(f"Diferença (linhas - IDs únicos): {len(df) - ids_unicos}")


Linhas totais: 5.250
Duplicidades completas: 60
Linhas envolvidas em duplicidade de Customer ID: 360
Customer IDs únicos: 5.070
Diferença (linhas - IDs únicos): 180


**Interpretação:** a diferença entre o total de linhas e a quantidade de
`Customer ID` únicos revela quantos registros "extras" existem — uma mistura
de duplicidades completas (provavelmente reenvios do mesmo arquivo) e
duplicidades por chave (o mesmo cliente atualizado por sistemas diferentes,
com pequenas divergências). O tratamento definitivo é feito no notebook 02,
mantendo sempre o registro mais recente por `Customer ID`.

Além disso, nomes muito parecidos sob `Customer ID` **diferentes** podem
indicar o mesmo cliente cadastrado duas vezes por engano — isso não aparece
nas contagens acima e será investigado separadamente com *fuzzy matching*
(RapidFuzz) no notebook 02, sem remoção automática.


## Cardinalidade

Quantos valores distintos cada coluna assume — útil para identificar colunas
categóricas "sujas" (muitos valores distintos que deveriam ser poucos, como
`Gender` e `State`).


In [6]:
df.nunique().sort_values(ascending=False)


Customer ID           5070
Email                 5045
Customer Name         4909
Income                4146
Total Spent           3807
Signup Date           3415
Age                     68
State                   62
Purchase Count          62
City                    44
Satisfaction Score      37
Gender                  17
dtype: int64

**Interpretação:** `Gender` deveria assumir poucas categorias (masculino,
feminino, outro), mas a cardinalidade observada é bem maior — sinal claro de
inconsistência de digitação/formato (`M`, `Masc`, `male`, `MASCULINO`...).
O mesmo vale para `State`, que deveria ter no máximo ~27 valores (as UFs
brasileiras) e apresenta uma cardinalidade maior, misturando sigla e nome
completo do estado, com e sem acentuação.


In [7]:
print("Valores distintos de Gender:", sorted(df["Gender"].dropna().unique()))


Valores distintos de Gender: ['F', 'FEMININO', 'Fem', 'Feminino', 'M', 'MASCULINO', 'Masc', 'Masculino', 'N/I', 'Não binário', 'Outro', 'fem', 'female', 'male', 'masc', 'outro', 'prefer not to say']


In [8]:
print("Valores distintos de State:", sorted(df["State"].dropna().unique()))


Valores distintos de State: ['AM', 'Amazonas', 'BA', 'BA ', 'Bahia', 'CE', 'Ceara', 'Ceará', 'DF', 'Distrito Federal', 'ES', 'Espirito Santo', 'Espírito Santo', 'GO', 'Goias', 'Goiás', 'MG', 'MT', 'Mato Grosso', 'Minas Gerais', 'PA', 'PE', 'PE ', 'PR', 'Para', 'Parana', 'Paraná', 'Pará', 'Pernambuco', 'RJ', 'RJ ', 'RS', 'Rio De Janeiro', 'Rio Grande do Sul', 'Rio de Janeiro', 'SC', 'SP', 'Santa Catarina', 'Sao Paulo', 'SÃO PAULO', 'São Paulo', 'am', 'amazonas', 'bahia', 'ce', 'df', 'distrito federal', 'es', 'go', 'mato grosso', 'mg', 'minas gerais', 'mt', 'pa', 'pernambuco', 'pr', 'rio de janeiro', 'rio grande do sul', 'rs', 'santa catarina', 'sc', 'sp']


## Estatísticas iniciais (`describe(include="all")`)

Mesmo com os tipos ainda incorretos (texto em vez de número), o `describe`
já revela pistas importantes: a alta cardinalidade de `Income`/`Total Spent`
(quase um valor distinto por linha, esperado para variáveis contínuas) e a
presença de valores "estranhos" nas colunas numéricas armazenadas como texto.


In [9]:
df.describe(include="all").T


,count,unique,top,freq
Customer ID,5250,5070,CUST001385,2
Customer Name,5250,4909,felipe da luz,3
Email,5250,5045,LAIS.DA.LUZ@MAIL.COM,3
Age,5043,68,18.0,319
Gender,5250,17,male,468
City,4995,44,Ribeirão Preto,235
State,5250,62,sp,256
Signup Date,5250,3415,2024/06/25,8
Income,4942,4146,900,22
Purchase Count,5250,62,7,784


## Amostra de idades fora do intervalo válido (18-100)

Um exemplo de problema que só aparece ao olhar os *valores*, não apenas os
tipos: idades logicamente impossíveis.


In [10]:
idade_numerica = pd.to_numeric(df["Age"], errors="coerce")
idades_invalidas = df.loc[idade_numerica.notna() & ~idade_numerica.between(18, 100), "Age"]
print(f"Registros com idade fora de 18-100: {len(idades_invalidas)}")
idades_invalidas.value_counts()


Registros com idade fora de 18-100: 77


Age
150.0    24
999.0    20
0.0      18
-5.0     15
Name: count, dtype: int64

## Resumo dos principais problemas encontrados (antes da limpeza)

Com base no profiling acima, os problemas de qualidade identificados na base
RAW são:

1. **Tipos incorretos**: colunas numéricas (`Age`, `Income`, `Purchase Count`,
   `Total Spent`, `Satisfaction Score`) e de data (`Signup Date`) armazenadas
   como texto.
2. **Valores ausentes** concentrados em `Age`, `City`, `Income` e
   `Satisfaction Score` (não em todas as colunas).
3. **Duplicidades** completas, por `Customer ID` e aproximadas (fuzzy, mesma
   pessoa sob IDs diferentes).
4. **Categorias inconsistentes** em `Gender` e `State` (múltiplas grafias
   para o mesmo valor).
5. **E-mails inconsistentes** (maiúsculas, espaços extras).
6. **Valores monetários em formato de texto** (`"4500"`, `"5.200"`,
   `"R$ 6.300,00"` — três formatos diferentes para a mesma grandeza).
7. **Datas em múltiplos formatos** (`YYYY-MM-DD`, `DD/MM/YYYY`, `DD-MM-YYYY`,
   `YYYY/MM/DD`).
8. **Idades inválidas** (negativas, zero ou irrealisticamente altas).
9. **Outliers extremos** em `Income`, `Total Spent` e `Purchase Count`
   (investigados estatisticamente no notebook 03).

O próximo notebook (`02_data_cleaning.ipynb`) trata cada um desses problemas
de forma isolada e documentada, usando as funções reutilizáveis do módulo
`src/data_cleaning.py`.
